In [8]:

# Data to store in JSON file
import json

import pandas as pd
from h11 import PRODUCT_ID

data = [
    {
        "id": 1,
        "name": "Sunday Joshi",
        "age": 22,
        "department": "Social Science",
        "salary": 6000
    },
    {
        "id": 2,
        "name": "Sita Sharma",
        "age": 25,
        "department": "Engineer",
        "salary": 5000
    },
    {
        "id": 3,
        "name": "Ram Karki",
        "age": 30,
        "department": "Front Desk",
        "salary": 5000
    }
]

with open ("employees.json", "w") as file:
    json.dump(data, file, indent=4)
print ("'employees.json' created.")

df = pd.read_json("employees.json")
print("Loaded DataFrame:")

df


'employees.json' created.
Loaded DataFrame:


,id,name,age,department,salary
0,1,Sunday Joshi,22,Social Science,6000
1,2,Sita Sharma,25,Engineer,5000
2,3,Ram Karki,30,Front Desk,5000


In [9]:
df.set_index("id", inplace=True)
df.head()
df


,name,age,department,salary
id,,,,
1,Sunday Joshi,22,Social Science,6000
2,Sita Sharma,25,Engineer,5000
3,Ram Karki,30,Front Desk,5000


In [10]:
# ## 3. The Data Cleaning Workflow
#
# ### Step 1: Handling Missing Values
#
# Missing data is often represented as `NaN` (Not a Number). Our first step is to identify where they are and decide on a strategy.
#
# **Strategy Options:**
# 1.  **Drop:** Remove rows or columns with missing values. (Use if the data is unusable or if you have a huge dataset and can afford to lose some rows).
# 2.  **Fill (Impute):** Replace missing values with something meaningful (e.g., 0, the mean, the median, or the most frequent value).
#
# First, let's count the `NaN`s in each column.

In [11]:
messy_data_csv = """OrderID,OrderDate,Product,Price,Quantity,Region
1001,2023-01-05,Laptop,$100,2,North
1002,2023-01-07,Mouse,$25.50,5,South
1003,2023-01-10,Keyboard,,3,North
1004,2023-01-12,Monitor,$300,,"West"
1005,2023-01-15,Webcam,$45.99,1,East
1002,2023-01-07,Mouse,$25.50,5,South
1006,2023-01-18,,$15.00,2,East
1007,2023-01-20,Laptop,$1200.00,1, North
1008,2023-01-22,External HDD,$80,4,USA
"""
#count missing values in each column
with open ("sales_data_messy.csv","w") as file:
    file.write(messy_data_csv)
print("'sales_data_messy.csv' created.")



'sales_data_messy.csv' created.


In [12]:
df = pd.read_csv("sales_data_messy.csv")
print("Missing values in each column:")
df.isnull()

Missing values in each column:


,OrderID,OrderDate,Product,Price,Quantity,Region
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,True,False,False
3,False,False,False,False,True,False
4,False,False,False,False,False,False
5,False,False,False,False,False,False
6,False,False,True,False,False,False
7,False,False,False,False,False,False
8,False,False,False,False,False,False


In [13]:
df.isnull().sum()

OrderID      0
OrderDate    0
Product      1
Price        1
Quantity     1
Region       0
dtype: int64

In [14]:
# #### Cleaning `Product` and `Quantity`
#
# *   The missing `Product` name makes that row less useful for sales analysis.
# *   A missing `Quantity` could be assumed to be 0, but a missing price is harder to guess.
#
# Let's start by filling the missing `Quantity` with the **median** value of the column. Using the median is often better than the mean because it's less sensitive to outliers.
#
# And for the missing `Product`, we will fill it with the string 'Unknown'.

In [15]:
median_quantity = df['Quantity'].median()
print(F"Median Quantity: {median_quantity}")
df.fillna({'Quantity': median_quantity, 'Product': 'Unknown'}, inplace=True)



Median Quantity: 2.5


In [16]:
df

,OrderID,OrderDate,Product,Price,Quantity,Region
0,1001,2023-01-05,Laptop,$100,2.0,North
1,1002,2023-01-07,Mouse,$25.50,5.0,South
2,1003,2023-01-10,Keyboard,NaN,3.0,North
3,1004,2023-01-12,Monitor,$300,2.5,West
4,1005,2023-01-15,Webcam,$45.99,1.0,East
5,1002,2023-01-07,Mouse,$25.50,5.0,South
6,1006,2023-01-18,Unknown,$15.00,2.0,East
7,1007,2023-01-20,Laptop,$1200.00,1.0,North
8,1008,2023-01-22,External HDD,$80,4.0,USA


In [17]:
print("\nMissing values after filling Quantity and Product:")
df.isnull().sum()


Missing values after filling Quantity and Product:


OrderID      0
OrderDate    0
Product      0
Price        1
Quantity     0
Region       0
dtype: int64

In [18]:
df["Price"]=df["Price"].str.replace("$","",regex=False)
df


,OrderID,OrderDate,Product,Price,Quantity,Region
0,1001,2023-01-05,Laptop,100,2.0,North
1,1002,2023-01-07,Mouse,25.50,5.0,South
2,1003,2023-01-10,Keyboard,NaN,3.0,North
3,1004,2023-01-12,Monitor,300,2.5,West
4,1005,2023-01-15,Webcam,45.99,1.0,East
5,1002,2023-01-07,Mouse,25.50,5.0,South
6,1006,2023-01-18,Unknown,15.00,2.0,East
7,1007,2023-01-20,Laptop,1200.00,1.0,North
8,1008,2023-01-22,External HDD,80,4.0,USA


In [19]:
df["Price"]=df["Price"].astype(float)


In [20]:
df

,OrderID,OrderDate,Product,Price,Quantity,Region
0,1001,2023-01-05,Laptop,100.00,2.0,North
1,1002,2023-01-07,Mouse,25.50,5.0,South
2,1003,2023-01-10,Keyboard,NaN,3.0,North
3,1004,2023-01-12,Monitor,300.00,2.5,West
4,1005,2023-01-15,Webcam,45.99,1.0,East
5,1002,2023-01-07,Mouse,25.50,5.0,South
6,1006,2023-01-18,Unknown,15.00,2.0,East
7,1007,2023-01-20,Laptop,1200.00,1.0,North
8,1008,2023-01-22,External HDD,80.00,4.0,USA


In [21]:
df.dtypes


OrderID        int64
OrderDate     object
Product       object
Price        float64
Quantity     float64
Region        object
dtype: object

In [22]:
mean_quantity = df['Price'].mean()
print(F"Mean Price: {mean_quantity}")
df.fillna({'Price': mean_quantity, 'Product': 'Unknown'}, inplace=True)



Mean Price: 223.99875


In [23]:
df


,OrderID,OrderDate,Product,Price,Quantity,Region
0,1001,2023-01-05,Laptop,100.00000,2.0,North
1,1002,2023-01-07,Mouse,25.50000,5.0,South
2,1003,2023-01-10,Keyboard,223.99875,3.0,North
3,1004,2023-01-12,Monitor,300.00000,2.5,West
4,1005,2023-01-15,Webcam,45.99000,1.0,East
5,1002,2023-01-07,Mouse,25.50000,5.0,South
6,1006,2023-01-18,Unknown,15.00000,2.0,East
7,1007,2023-01-20,Laptop,1200.00000,1.0,North
8,1008,2023-01-22,External HDD,80.00000,4.0,USA


In [24]:

df['Total_Sales'] = df['Price'] * df['Quantity']
df


,OrderID,OrderDate,Product,Price,Quantity,Region,Total_Sales
0,1001,2023-01-05,Laptop,100.00000,2.0,North,200.00000
1,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
2,1003,2023-01-10,Keyboard,223.99875,3.0,North,671.99625
3,1004,2023-01-12,Monitor,300.00000,2.5,West,750.00000
4,1005,2023-01-15,Webcam,45.99000,1.0,East,45.99000
5,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
6,1006,2023-01-18,Unknown,15.00000,2.0,East,30.00000
7,1007,2023-01-20,Laptop,1200.00000,1.0,North,1200.00000
8,1008,2023-01-22,External HDD,80.00000,4.0,USA,320.00000


In [25]:
df['Total_sale']=df['Price']*df['Quantity']
df

,OrderID,OrderDate,Product,Price,Quantity,Region,Total_Sales,Total_sale
0,1001,2023-01-05,Laptop,100.00000,2.0,North,200.00000,200.00000
1,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000,127.50000
2,1003,2023-01-10,Keyboard,223.99875,3.0,North,671.99625,671.99625
3,1004,2023-01-12,Monitor,300.00000,2.5,West,750.00000,750.00000
4,1005,2023-01-15,Webcam,45.99000,1.0,East,45.99000,45.99000
5,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000,127.50000
6,1006,2023-01-18,Unknown,15.00000,2.0,East,30.00000,30.00000
7,1007,2023-01-20,Laptop,1200.00000,1.0,North,1200.00000,1200.00000
8,1008,2023-01-22,External HDD,80.00000,4.0,USA,320.00000,320.00000


In [26]:
df.drop('Total_sale',axis=1,inplace=True)
df

,OrderID,OrderDate,Product,Price,Quantity,Region,Total_Sales
0,1001,2023-01-05,Laptop,100.00000,2.0,North,200.00000
1,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
2,1003,2023-01-10,Keyboard,223.99875,3.0,North,671.99625
3,1004,2023-01-12,Monitor,300.00000,2.5,West,750.00000
4,1005,2023-01-15,Webcam,45.99000,1.0,East,45.99000
5,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
6,1006,2023-01-18,Unknown,15.00000,2.0,East,30.00000
7,1007,2023-01-20,Laptop,1200.00000,1.0,North,1200.00000
8,1008,2023-01-22,External HDD,80.00000,4.0,USA,320.00000


In [27]:
#sum of total sales
total_sales=df['Total_Sales'].sum()
print(f"Total Sales: {total_sales}")

Total Sales: 3472.98625


In [28]:
df.duplicated()


0    False
1    False
2    False
3    False
4    False
5     True
6    False
7    False
8    False
dtype: bool

In [29]:
df=df.drop_duplicates()

In [30]:
df

,OrderID,OrderDate,Product,Price,Quantity,Region,Total_Sales
0,1001,2023-01-05,Laptop,100.00000,2.0,North,200.00000
1,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
2,1003,2023-01-10,Keyboard,223.99875,3.0,North,671.99625
3,1004,2023-01-12,Monitor,300.00000,2.5,West,750.00000
4,1005,2023-01-15,Webcam,45.99000,1.0,East,45.99000
6,1006,2023-01-18,Unknown,15.00000,2.0,East,30.00000
7,1007,2023-01-20,Laptop,1200.00000,1.0,North,1200.00000
8,1008,2023-01-22,External HDD,80.00000,4.0,USA,320.00000


In [31]:
df.shape

(8, 7)

In [32]:
df.to_csv('sales_data.csv')

In [33]:
df=pd.read_csv('sales_data.csv')
df

,Unnamed: 0,OrderID,OrderDate,Product,Price,Quantity,Region,Total_Sales
0,0,1001,2023-01-05,Laptop,100.00000,2.0,North,200.00000
1,1,1002,2023-01-07,Mouse,25.50000,5.0,South,127.50000
2,2,1003,2023-01-10,Keyboard,223.99875,3.0,North,671.99625
3,3,1004,2023-01-12,Monitor,300.00000,2.5,West,750.00000
4,4,1005,2023-01-15,Webcam,45.99000,1.0,East,45.99000
5,6,1006,2023-01-18,Unknown,15.00000,2.0,East,30.00000
6,7,1007,2023-01-20,Laptop,1200.00000,1.0,North,1200.00000
7,8,1008,2023-01-22,External HDD,80.00000,4.0,USA,320.00000


In [34]:
import os
print(os.getcwd())


/Users/danuzz/Documents/notebooks/learnPythonnnnnnnnnn/jupyternotebook/pandas


In [35]:
new_df=df[['Product','Price','Quantity']]
new_df

,Product,Price,Quantity
0,Laptop,100.00000,2.0
1,Mouse,25.50000,5.0
2,Keyboard,223.99875,3.0
3,Monitor,300.00000,2.5
4,Webcam,45.99000,1.0
5,Unknown,15.00000,2.0
6,Laptop,1200.00000,1.0
7,External HDD,80.00000,4.0


In [36]:
new_df.to_csv('Product_details.csv')

In [37]:
a=pd.read_csv('Product_details.csv')
a

,Unnamed: 0,Product,Price,Quantity
0,0,Laptop,100.00000,2.0
1,1,Mouse,25.50000,5.0
2,2,Keyboard,223.99875,3.0
3,3,Monitor,300.00000,2.5
4,4,Webcam,45.99000,1.0
5,5,Unknown,15.00000,2.0
6,6,Laptop,1200.00000,1.0
7,7,External HDD,80.00000,4.0


In [38]:
x=1
def f():
    global x
    x+=2
    return x
f()
print(f())


5


'ecommerce_sales_data.csv' created.


,OrderID,OrderDate,Product,Price,Quantity,Region
0,1001,2023-01-05,Laptop,$100,2.0,North
1,1002,2023-01-07,Mouse,$25.50,5.0,South
2,1003,2023-01-10,Keyboard,NaN,3.0,North
3,1004,2023-01-12,Monitor,$300,NaN,West
4,1005,2023-01-15,Webcam,$45.99,1.0,East
5,1002,2023-01-07,Mouse,$25.50,5.0,South
6,1006,2023-01-18,NaN,$15.00,2.0,East
7,1007,2023-01-20,Laptop,$1200.00,1.0,North
8,1008,2023-01-22,External HDD,$80,4.0,USA
